# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [1]:
import os
import random
from copy import deepcopy

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split

# optional / existing
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [2]:
from pathlib import Path

DATA_DIR = Path.cwd() / "Hackathon_data"
DATA_DIR

PosixPath('/Users/swapnilmittal/Downloads/MLB_Hackathon/Hackathon_data')

In [3]:
import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'xgboost', 'optuna'])

0

In [ ]:
with open(f"{DATA_DIR}/sequence.fasta", "r") as f:
    data = f.readlines()

sequence_wt = data[1].strip()
print(sequence_wt[:20], "...")

MVNEARGNSSLNPCLEGSAS ...


In [5]:
len(sequence_wt)

656

In [6]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [7]:
df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
original_train_mutants = set(df_train['mutant'])
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [8]:
df_test = pd.read_csv(f"{DATA_DIR}/test.csv")
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [9]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [10]:
'''hyperparameters'''

seq_length = 656
seed = 0  # seed for splitting the validation set
val_ratio = 0.2  # proportion of validation set
checkpoint_threshold = 0.02

In [11]:
from Bio.Align import substitution_matrices
blosum62 = substitution_matrices.load("BLOSUM62")

AA_PROPS = {
    'A': [1.8,   0,   89,  67],
    'C': [2.5,   0,  121,  86],
    'D': [-3.5, -1,  133,  91],
    'E': [-3.5, -1,  147, 109],
    'F': [2.8,   0,  165, 135],
    'G': [-0.4,  0,   75,  48],
    'H': [-3.2,  0.5, 155, 118],
    'I': [4.5,   0,  131, 124],
    'K': [-3.9,  1,  146, 135],
    'L': [3.8,   0,  131, 124],
    'M': [1.9,   0,  149, 124],
    'N': [-3.5,  0,  132,  96],
    'P': [-1.6,  0,  115,  90],
    'Q': [-3.5,  0,  146, 114],
    'R': [-4.5,  1,  174, 148],
    'S': [-0.8,  0,  105,  73],
    'T': [-0.7,  0,  119,  93],
    'V': [4.2,   0,  117, 105],
    'W': [-0.9,  0,  204, 163],
    'Y': [-1.3,  0,  181, 141],
}


def encode_mutant(mutant, seq_length=656):
    wildtype_aa  = mutant[0]
    position     = int(mutant[1:-1])
    mutant_aa    = mutant[-1]

    wildtype_props  = np.array(AA_PROPS[wildtype_aa])
    mutant_props    = np.array(AA_PROPS[mutant_aa])
    prop_delta      = mutant_props - wildtype_props

    alphabet = 'ACDEFGHIKLMNPQRSTVWY'
    wildtype_onehot = np.zeros(20); wildtype_onehot[alphabet.index(wildtype_aa)] = 1
    mutant_onehot   = np.zeros(20); mutant_onehot[alphabet.index(mutant_aa)]     = 1

    normalized_position = position / seq_length
    blosum_score        = blosum62[wildtype_aa][mutant_aa]

    # 46-dim embedding:
    # [position(1),wildtype_aa(20),mutant_aa(20),physicochemical_delta(4),blosum62(1)]
    mutation_embedding = np.concatenate([
        [normalized_position],
        wildtype_onehot,
        mutant_onehot,
        prop_delta,
        [blosum_score],
    ]).astype(np.float32)

    return mutation_embedding


X_all   = np.stack(df_train['mutant'].apply(encode_mutant).values)
X_test  = np.stack(df_test['mutant'].apply(encode_mutant).values)
y_all   = df_train['DMS_score'].values

train_idx, val_idx = train_test_split(range(len(df_train)), test_size=val_ratio, random_state=seed)
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val,   y_val   = X_all[val_idx],   y_all[val_idx]

print(f"Features: {X_train.shape[1]}, Train: {len(X_train)}, Val: {len(X_val)}")


Features: 46, Train: 912, Val: 228


In [12]:
from xgboost import XGBRegressor
import optuna

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'early_stopping_rounds': 30,
        'random_state':      seed,
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_pred = model.predict(X_val)
    return spearmanr(y_val, val_pred)[0]

sampler = optuna.samplers.TPESampler(seed=seed)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best Spearman: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

best_model = XGBRegressor(**study.best_params, early_stopping_rounds=30, random_state=seed)
best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
val_sp, _ = spearmanr(y_val, best_model.predict(X_val))
print(f"Val Spearman with best params: {val_sp:.4f}")
if val_sp >= checkpoint_threshold:
    print(f"Checkpoint threshold passed on validation split (>= {checkpoint_threshold:.2f}).")
else:
    print(f"Checkpoint threshold NOT passed on validation split (< {checkpoint_threshold:.2f}).")

y_test_pred = best_model.predict(X_test)


[I 2026-04-06 00:57:17,593] A new study created in memory with name: no-name-62d60288-118b-4638-941a-629660bd301d


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-06 00:57:17,673] Trial 0 finished with value: 0.47717411389116743 and parameters: {'n_estimators': 1143, 'learning_rate': 0.11387317092169064, 'max_depth': 6, 'subsample': 0.7724415914984484, 'colsample_bytree': 0.7118273996694524, 'min_child_weight': 7}. Best is trial 0 with value: 0.47717411389116743.
[I 2026-04-06 00:57:17,711] Trial 1 finished with value: 0.4245794133256151 and parameters: {'n_estimators': 931, 'learning_rate': 0.20761410420015303, 'max_depth': 8, 'subsample': 0.6917207594128889, 'colsample_bytree': 0.8958625190413323, 'min_child_weight': 6}. Best is trial 0 with value: 0.47717411389116743.
[I 2026-04-06 00:57:17,739] Trial 2 finished with value: 0.4976702733049873 and parameters: {'n_estimators': 1179, 'learning_rate': 0.2329262676497442, 'max_depth': 3, 'subsample': 0.5435646498507704, 'colsample_bytree': 0.5101091987201629, 'min_child_weight': 9}. Best is trial 2 with value: 0.4976702733049873.
[I 2026-04-06 00:57:17,785] Trial 3 finished with value: 

[I 2026-04-06 00:57:17,915] Trial 5 finished with value: 0.4984798830990939 and parameters: {'n_estimators': 602, 'learning_rate': 0.13919929598274874, 'max_depth': 5, 'subsample': 0.7842169744343243, 'colsample_bytree': 0.5093949002181776, 'min_child_weight': 7}. Best is trial 4 with value: 0.5054250058294119.
[I 2026-04-06 00:57:17,969] Trial 6 finished with value: 0.4797204942546897 and parameters: {'n_estimators': 1263, 'learning_rate': 0.0815241575370242, 'max_depth': 8, 'subsample': 0.8409101495517417, 'colsample_bytree': 0.679753950286893, 'min_child_weight': 5}. Best is trial 4 with value: 0.5054250058294119.


[I 2026-04-06 00:57:18,161] Trial 7 finished with value: 0.48510932133016005 and parameters: {'n_estimators': 1426, 'learning_rate': 0.012273271012844034, 'max_depth': 7, 'subsample': 0.8353189348090797, 'colsample_bytree': 0.6051912805369204, 'min_child_weight': 2}. Best is trial 4 with value: 0.5054250058294119.
[I 2026-04-06 00:57:18,241] Trial 8 finished with value: 0.4999715239563673 and parameters: {'n_estimators': 699, 'learning_rate': 0.03445441737078129, 'max_depth': 6, 'subsample': 0.7193007567311602, 'colsample_bytree': 0.9941869190296131, 'min_child_weight': 2}. Best is trial 4 with value: 0.5054250058294119.


...

[I 2026-04-06 00:57:20,027] Trial 29 finished with value: 0.49769023252031286 and parameters: {'n_estimators': 1761, 'learning_rate': 0.2885580328726487, 'max_depth': 7, 'subsample': 0.6516354589892395, 'colsample_bytree': 0.7057990467256525, 'min_child_weight': 7}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,083] Trial 30 finished with value: 0.5151676074750227 and parameters: {'n_estimators': 1064, 'learning_rate': 0.12639478668899393, 'max_depth': 4, 'subsample': 0.7317772556972783, 'colsample_bytree': 0.7651850117219609, 'min_child_weight': 2}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,139] Trial 31 finished with value: 0.5059968051139428 and parameters: {'n_estimators': 1023, 'learning_rate': 0.09998604603453472, 'max_depth': 4, 'subsample': 0.7206861326250634, 'colsample_bytree': 0.7690631637144921, 'min_child_weight': 2}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,191] Trial 32 finished with v

[I 2026-04-06 00:57:20,245] Trial 33 finished with value: 0.4836291308121053 and parameters: {'n_estimators': 1245, 'learning_rate': 0.06508296161530916, 'max_depth': 3, 'subsample': 0.7785276760898648, 'colsample_bytree': 0.7481721273401798, 'min_child_weight': 2}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,302] Trial 34 finished with value: 0.47604266596687556 and parameters: {'n_estimators': 739, 'learning_rate': 0.11763338091506599, 'max_depth': 5, 'subsample': 0.5622323650739713, 'colsample_bytree': 0.6329897853715056, 'min_child_weight': 9}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,341] Trial 35 finished with value: 0.49936099624357655 and parameters: {'n_estimators': 1362, 'learning_rate': 0.18925063599547126, 'max_depth': 3, 'subsample': 0.7006104405974999, 'colsample_bytree': 0.705763725793798, 'min_child_weight': 1}. Best is trial 21 with value: 0.5237155195700736.
[I 2026-04-06 00:57:20,394] Trial 36 finished with v

[I 2026-04-06 00:57:20,459] Trial 37 finished with value: 0.5245677523948058 and parameters: {'n_estimators': 560, 'learning_rate': 0.09965956548376258, 'max_depth': 3, 'subsample': 0.6070197448946952, 'colsample_bytree': 0.857279264550014, 'min_child_weight': 8}. Best is trial 37 with value: 0.5245677523948058.
[I 2026-04-06 00:57:20,496] Trial 38 finished with value: 0.4909540598445676 and parameters: {'n_estimators': 559, 'learning_rate': 0.1181884877492464, 'max_depth': 3, 'subsample': 0.8121985703645673, 'colsample_bytree': 0.8675282254736209, 'min_child_weight': 9}. Best is trial 37 with value: 0.5245677523948058.
[I 2026-04-06 00:57:20,542] Trial 39 finished with value: 0.5265970870309232 and parameters: {'n_estimators': 886, 'learning_rate': 0.17850730930518635, 'max_depth': 3, 'subsample': 0.6657237855488258, 'colsample_bytree': 0.9371670562012859, 'min_child_weight': 8}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:20,579] Trial 40 finished with value:

[I 2026-04-06 00:57:20,662] Trial 42 finished with value: 0.49891271964912876 and parameters: {'n_estimators': 1113, 'learning_rate': 0.18011784228376299, 'max_depth': 3, 'subsample': 0.710709616482613, 'colsample_bytree': 0.7318275843758917, 'min_child_weight': 8}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:20,702] Trial 43 finished with value: 0.49546027059679176 and parameters: {'n_estimators': 809, 'learning_rate': 0.159622160601696, 'max_depth': 3, 'subsample': 0.7852121684654323, 'colsample_bytree': 0.8439265874518267, 'min_child_weight': 6}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:20,743] Trial 44 finished with value: 0.4963107710321602 and parameters: {'n_estimators': 889, 'learning_rate': 0.28344632821664845, 'max_depth': 3, 'subsample': 0.6504497429281461, 'colsample_bytree': 0.6821632307487921, 'min_child_weight': 8}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:20,786] Trial 45 finished with valu

[I 2026-04-06 00:57:20,876] Trial 47 finished with value: 0.5136368859860247 and parameters: {'n_estimators': 1177, 'learning_rate': 0.10219220628245565, 'max_depth': 3, 'subsample': 0.6042051671939992, 'colsample_bytree': 0.9731985709316603, 'min_child_weight': 9}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:20,953] Trial 48 finished with value: 0.48049974042775734 and parameters: {'n_estimators': 1529, 'learning_rate': 0.07772658883272858, 'max_depth': 7, 'subsample': 0.6163051108245623, 'colsample_bytree': 0.9143908066278768, 'min_child_weight': 8}. Best is trial 39 with value: 0.5265970870309232.
[I 2026-04-06 00:57:21,037] Trial 49 finished with value: 0.5169128812293404 and parameters: {'n_estimators': 1367, 'learning_rate': 0.106691001385786, 'max_depth': 3, 'subsample': 0.5339376593653163, 'colsample_bytree': 0.8265161415701654, 'min_child_weight': 8}. Best is trial 39 with value: 0.5265970870309232.
Best Spearman: 0.5266
Best params: {'n_estimators': 8

In [13]:
df_test = df_test.copy()
df_test['DMS_score_predicted'] = y_test_pred
df_test[['mutant', 'DMS_score_predicted']].head()

,mutant,DMS_score_predicted
0,V1D,0.259159
1,V1Y,0.247830
2,V1C,0.365254
3,V1A,0.435335
4,V1E,0.252548


In [14]:
submission_df = df_test[['mutant', 'DMS_score_predicted']].copy()
submission_df.to_csv('predictions.csv', index=False)
submission_df.to_csv('test_predictions.csv', index=False)
submission_df.head()

,mutant,DMS_score_predicted
0,V1D,0.259159
1,V1Y,0.247830
2,V1C,0.365254
3,V1A,0.435335
4,V1E,0.252548


## 3. Select query for next round

In [15]:
top10_df = (
    df_test.loc[~df_test['mutant'].isin(original_train_mutants), ['mutant', 'DMS_score_predicted']]
    .sort_values('DMS_score_predicted', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10_df

,mutant,DMS_score_predicted
0,V535I,0.778711
1,V527I,0.778711
2,V558I,0.778711
3,V542I,0.778711
4,V430I,0.778711
5,V361I,0.778711
6,V473I,0.778711
7,V540I,0.778711
8,V433I,0.778711
9,V573I,0.778711


In [16]:
with open('top10.txt', 'w') as f:
    for mutant in top10_df['mutant']:
        f.write(f"{mutant}\n")

print('Saved top10.txt')
top10_df['mutant'].tolist()


Saved top10.txt


['V535I',
 'V527I',
 'V558I',
 'V542I',
 'V430I',
 'V361I',
 'V473I',
 'V540I',
 'V433I',
 'V573I']

In [17]:
assert len(submission_df) == len(pd.read_csv(f"{DATA_DIR}/test.csv")), 'predictions file must cover all test mutants'
assert submission_df['mutant'].is_unique, 'predictions contain duplicate mutants'
assert len(top10_df) == 10, 'top10.txt must contain exactly 10 mutants'
assert top10_df['mutant'].is_unique, 'top10 contains duplicate mutants'
assert set(top10_df['mutant']).issubset(set(df_test['mutant'])), 'top10 mutants must come from test.csv'
assert set(top10_df['mutant']).isdisjoint(original_train_mutants), 'top10 cannot include original train.csv mutants'

print('Submission sanity checks passed.')

Submission sanity checks passed.


In [18]:
print('Final files written: predictions.csv, test_predictions.csv, top10.txt')
print('\nTop 10 recommended mutants:')
for mutant in top10_df['mutant']:
    print(mutant)

Final files written: predictions.csv, test_predictions.csv, top10.txt

Top 10 recommended mutants:
V535I
V527I
V558I
V542I
V430I
V361I
V473I
V540I
V433I
V573I


In [19]:
pd.read_csv('predictions.csv').head()

,mutant,DMS_score_predicted
0,V1D,0.259159
1,V1Y,0.247830
2,V1C,0.365254
3,V1A,0.435335
4,V1E,0.252548
